# Frozen s1-small inference on a Colab T4

Use **Runtime → Change runtime type → T4 GPU**. Training stays completed on the Mac. This notebook runs a separate CUDA replication using the exact packed Q4/group-64 base, adapter and FP32 recipe. No model training occurs.

Push the small code repository yourself, upload `nlp-colab-assets.zip` to your own Drive under `MyDrive/NLP-Colab/`, then fill `REPO_URL` below. The asset ZIP stays outside GitHub. It selects the exact source commit recorded when the bundle was built. A full `--include-code` bundle can instead be used with an empty repo URL.

**CUDA is not yet tested on your allocated GPU.** Stop if verification or smoke fails. Free GPU allocation/runtime is not guaranteed; save checkpoints and resume in another permitted session. Do not merge CUDA outputs/timings with the running Mac experiment.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, subprocess, json, zipfile, hashlib, time, os
assert sys.platform.startswith('linux'), 'Use a Linux Colab runtime'
assert sys.version_info >= (3, 11), 'Pinned NumPy requires Python 3.11+'
subprocess.run(['nvidia-smi'], check=True)
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/faisalAI27/Qwen2.5-1.5B-finetune.git'  # fill after you push code
BUNDLE = Path('/content/drive/MyDrive/NLP-Colab/nlp-colab-assets.zip')
DRIVE_STATE = Path('/content/drive/MyDrive/NLP-Colab/cuda-state')
ROOT = Path('/content/nlp-colab')
VENV = Path('/content/nlp-colab-env')


In [ ]:
assert BUNDLE.exists(), f'Upload the asset bundle to {BUNDLE}'
with zipfile.ZipFile(BUNDLE) as archive:
    bundled = archive.read('colab_bundle_manifest.json')
    metadata = json.loads(bundled)
    if metadata['bundle_mode'] == 'assets_with_github_code':
        assert 'YOUR_USER' not in REPO_URL and REPO_URL, 'Set your GitHub repository URL first'
        assert metadata['source_commit'], 'Bundle must be built after committing code'
        if not (ROOT / '.git').exists():
            subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
        subprocess.run(['git', 'checkout', '--detach', metadata['source_commit']], cwd=ROOT, check=True)
    ROOT.mkdir(parents=True, exist_ok=True)
    old = ROOT / 'colab_bundle_manifest.json'
    assert not old.exists() or old.read_bytes() == bundled, 'Use a fresh ROOT/state folder for a different bundle'
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        assert not relative.is_absolute() and '..' not in relative.parts, 'Unsafe archive path'
    archive.extractall(ROOT)
# A GitHub clone may contain the Mac freeze. Preserve it separately here only.
original = ROOT / 'manifests/evaluation_freeze_macos.json'
canonical = ROOT / 'manifests/evaluation_freeze.json'
if canonical.exists() and canonical.read_bytes() == original.read_bytes():
    canonical.rename(ROOT / 'manifests/evaluation_freeze_macos_from_clone.json')
subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(ROOT / 'requirements-colab.txt')], check=True)
subprocess.run([PYTHON, str(ROOT / 'scripts/colab_checkpoint.py'), 'restore', '--drive-folder', str(DRIVE_STATE)], check=True)
subprocess.run([PYTHON, str(ROOT / 'scripts/colab_inference.py'), 'verify'], check=True)


## Run and persist progress

The helper below snapshots mutable CUDA state to Drive every five minutes and after each stage, retaining its last two snapshots. The model stays on the VM's local disk. If a free session ends, rerun setup, restore the latest checkpoint and rerun the unfinished stage. Completed saved answers are retained. An in-flight answer or output since the last snapshot may need recomputation. If the GPU type/driver/runtime changes, the frozen identity rejects a mixed run; use a fresh state folder for a distinct hardware replication.


In [ ]:
def save_state():
    subprocess.run([PYTHON, str(ROOT / 'scripts/colab_checkpoint.py'), 'save', '--drive-folder', str(DRIVE_STATE)], check=True)

def run_stage(mode):
    log = ROOT / 'results' / ('colab-' + mode + '.log')
    log.parent.mkdir(parents=True, exist_ok=True)
    with log.open('ab', buffering=0) as stream:
        process = subprocess.Popen([PYTHON, str(ROOT / 'scripts/colab_inference.py'), mode], cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT)
        checkpoint_due = time.monotonic() + 300
        try:
            while process.poll() is None:
                time.sleep(30)
                print('\n'.join(log.read_text(errors='replace').splitlines()[-2:]))
                if time.monotonic() >= checkpoint_due:
                    save_state()
                    checkpoint_due = time.monotonic() + 300
        except BaseException:
            process.terminate()
            try: process.wait(timeout=20)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            save_state()
            raise
    save_state()
    print('\n'.join(log.read_text(errors='replace').splitlines()[-12:]))
    assert process.returncode == 0, f'{mode} stopped; evidence retained in {log}'


## 1. GPU compatibility smoke

Loads the exact adapter, checks its tensor fingerprint, tests a bounded held-out question and records memory/latency. This is not an accuracy test.


In [ ]:
run_stage('smoke')


## 2. Same 45 development outputs, then CUDA freeze

All nine conditions run on the same five held-out questions. The CUDA freeze records hardware/package identities and exact-token parity with Mac development outputs. Numerical differences are reported, not hidden or tuned against test answers. All CUDA benchmark conditions remain an independent replication.


In [ ]:
run_stage('development')
run_stage('freeze')


## 3. Full MATH-500 generation

Only the CUDA freeze permits benchmark answers to be downloaded. All 4,500 slots are explicit, including failures. Resume this cell after a session ends. No completed answer is retried because it is incorrect.


In [ ]:
run_stage('test')


## 4. Grade and analyze

These stages use the pinned local symbolic grader and can run without an allocated GPU. If a run remains partial, report the exact coverage; partial scores are not full benchmark accuracy. Keep outputs and GPU timings separate from Mac results.


In [ ]:
run_stage('grade')
run_stage('analyze')
subprocess.run([PYTHON, str(ROOT / 'scripts/colab_inference.py'), 'status'], check=True)


Results are in `results/test/math500-colab-cuda-v1/`, with `evaluated.jsonl`, analysis JSON and PNG/SVG plots. The latest resumable archive is in your selected Drive state folder. Preserve the CUDA freeze and recorded GPU hardware. The Mac experiment continues independently; importing this archive must use a separate directory.
